# Notebook #29 — Live Trades Replay vs Backtest (Bybit)

Compares the live bot's signals + entries + exits (recorded into
`logs/bybit_bot/<SYMBOL>-YYYY-MM-DD.jsonl`) against a **replay backtest** that
feeds the exact same winner config + the symbol's CSV data into
`_strategy_lib.run_strategy()`. Anything the live bot did that the replay didn't
(or vice versa) flags a parity bug.

## Inputs
- Live log lines       — `logs/bybit_bot/<SYM>-YYYY-MM-DD.jsonl`
- Per-symbol config    — `results/_top_per_symbol/<SYM>/config.json`
- Symbol price history — `notebooks/data/<SYM>/{M5,H1,D1}/ohlcv.csv`

## Outputs (`notebooks/data/`)
- `live_signals_<from>_<to>.csv`   — every signal the bot emitted in the window
- `live_orders_<from>_<to>.csv`    — every `market_order_placed` event
- `live_closes_<from>_<to>.csv`    — every `position_closed` event
- `replay_signals_<from>_<to>.csv` — signals the backtest would have produced
- `replay_vs_live_diff_<from>_<to>.csv` — joined view: matched, live-only, replay-only
- `replay_summary_<from>_<to>.csv` — per-symbol win-rate / net-R / parity stats

Set the window in **Section 1**; everything else is automatic.

## Workflow
1. Edit `WINDOW_FROM` / `WINDOW_TO` and (optionally) `SYMBOLS`.
2. Run all cells.
3. Inspect the `replay_vs_live_diff` table for matched / live-only / replay-only rows.
4. For each mismatch, look at the diagnostics in the live `cycle` log line
   (`logs/bybit_bot/<SYM>-YYYY-MM-DD.jsonl`) and the matching bar in the
   replay backtest's `df_sig` to see *which gate* disagreed.


In [1]:
from __future__ import annotations
import warnings; warnings.filterwarnings('ignore')
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:.6f}'.format)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    REPO = ROOT.parent
else:
    REPO = ROOT
NB_DIR  = REPO / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

from _strategy_lib import (
    run_strategy, stats,
    strategy_trend_pullback, strategy_bb_revert_midline,
    strategy_rsi_extreme_reversal, strategy_donchian_breakout,
    strategy_macd_pullback, strategy_ichimoku, strategy_fib_pullback,
    strategy_sr_zone_bounce, strategy_vwap_reaction, strategy_ema_cross,
)

STRATEGY_FN = {
    'trend_pullback':  strategy_trend_pullback,
    'bb_revert_mid':   strategy_bb_revert_midline,
    'rsi_extreme':     strategy_rsi_extreme_reversal,
    'donchian_brkout': strategy_donchian_breakout,
    'macd_pullback':   strategy_macd_pullback,
    'ichimoku':        strategy_ichimoku,
    'fib_pullback':    strategy_fib_pullback,
    'sr_zone_bounce':  strategy_sr_zone_bounce,
    'vwap_reaction':   strategy_vwap_reaction,
    'ema_cross':       strategy_ema_cross,
}
USE_DAILY = {'trend_pullback': True}

BOT_LOG_DIR = REPO / 'logs' / 'bybit_bot'
WINNERS_DIR = NB_DIR / 'results' / '_top_per_symbol'
DATA_DIR    = NB_DIR / 'data'
OUT_DIR     = NB_DIR / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────── EDIT THIS BLOCK ─────────────────
WINDOW_FROM = '2026-05-30'          # inclusive (UTC date)
WINDOW_TO   = '2026-12-31'          # exclusive (UTC date)
SYMBOLS     = None                  # None = every symbol that has a logs dir entry
BAR_TOLERANCE_MIN = 5               # treat replay vs live signals as 'matched'
                                    # when the bar times are within this many minutes
# ──────────────────────────────────────────────────

T_FROM = pd.Timestamp(WINDOW_FROM)
T_TO   = pd.Timestamp(WINDOW_TO)
TAG    = f"{WINDOW_FROM}_{WINDOW_TO}".replace('-', '')
print(f'Window: {WINDOW_FROM} → {WINDOW_TO}')
print(f'Bot logs dir: {BOT_LOG_DIR}')
print(f'Winners dir : {WINNERS_DIR}')


Window: 2026-05-30 → 2026-12-31
Bot logs dir: D:\bot\ema-h1trend-exchange\logs\bybit_bot
Winners dir : D:\bot\ema-h1trend-exchange\notebooks\results\_top_per_symbol


## Section 2 — Discover symbols that have either a live log or a winner config

We auto-pick the set of symbols based on what's on disk. Override via the `SYMBOLS`
constant above if you want a subset.

In [2]:
def list_logged_symbols() -> set[str]:
    if not BOT_LOG_DIR.exists():
        return set()
    syms = set()
    for p in BOT_LOG_DIR.glob('*.jsonl'):
        # filename pattern: SYMBOL-YYYY-MM-DD.jsonl  ; strip the date suffix
        stem = p.stem
        if stem.startswith('_'):
            continue
        parts = stem.rsplit('-', 3)
        sym = parts[0] if len(parts) >= 4 else stem
        syms.add(sym)
    return syms

def list_winner_symbols() -> set[str]:
    if not WINNERS_DIR.exists():
        return set()
    return {p.name for p in WINNERS_DIR.iterdir()
            if p.is_dir() and not p.name.startswith('_') and (p / 'config.json').exists()}

logged  = list_logged_symbols()
winners = list_winner_symbols()
active  = sorted(SYMBOLS) if SYMBOLS else sorted(logged | winners)
print(f'Symbols with live logs    : {len(logged):3d}  → {sorted(logged)}')
print(f'Symbols with winner config: {len(winners):3d}  → {sorted(winners)}')
print(f'Symbols to inspect        : {len(active):3d}  → {active}')


Symbols with live logs    :   2  → ['SOLUSDT', 'XLMUSDT']
Symbols with winner config:  14  → ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XAUUSDT', 'XLMUSDT', 'XRPUSDT']
Symbols to inspect        :  14  → ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XAUUSDT', 'XLMUSDT', 'XRPUSDT']


## Section 3 — Parse live JSONL logs

Each event lives on its own line. We extract three views:

- `signals`    — `event == 'signal'`
- `orders`     — `event == 'market_order_placed'`
- `closes`     — `event == 'position_closed'`

Each row carries the event timestamp + the diagnostics the bot wrote when it
decided. Multi-day logs are concatenated.

In [3]:
def parse_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows

def _ts_naive(x):
    """Parse any timestamp string/object to tz-naive UTC pd.Timestamp."""
    try:
        t = pd.Timestamp(x)
    except Exception:
        return pd.NaT
    if t is pd.NaT: return pd.NaT
    if t.tzinfo is not None:
        t = t.tz_convert('UTC').tz_localize(None)
    return t

def load_live(symbol: str, t_from: pd.Timestamp, t_to: pd.Timestamp) -> dict:
    sigs, orders, closes, cycles = [], [], [], []
    if not BOT_LOG_DIR.exists():
        return dict(signals=pd.DataFrame(), orders=pd.DataFrame(),
                    closes=pd.DataFrame(), cycles=pd.DataFrame())
    files = sorted(BOT_LOG_DIR.glob(f'{symbol}-*.jsonl'))
    for fp in files:
        for ev in parse_jsonl(fp):
            ts = _ts_naive(ev.get('ts', ''))
            if ts is pd.NaT or ts < t_from or ts >= t_to: continue
            kind = ev.get('event')
            ev['ts'] = ts
            if   kind == 'signal':                sigs.append(ev)
            elif kind == 'market_order_placed':   orders.append(ev)
            elif kind == 'position_closed':       closes.append(ev)
            elif kind == 'cycle':                 cycles.append(ev)
    return dict(
        signals=pd.DataFrame(sigs),
        orders=pd.DataFrame(orders),
        closes=pd.DataFrame(closes),
        cycles=pd.DataFrame(cycles),
    )

live_by_sym: dict[str, dict] = {}
for sym in active:
    live_by_sym[sym] = load_live(sym, T_FROM, T_TO)
n_total_sigs   = sum(len(v['signals']) for v in live_by_sym.values())
n_total_orders = sum(len(v['orders'])  for v in live_by_sym.values())
n_total_closes = sum(len(v['closes'])  for v in live_by_sym.values())
print(f'Loaded live events — signals={n_total_sigs}  orders={n_total_orders}  closes={n_total_closes}')
for sym in active:
    v = live_by_sym[sym]
    print(f'  {sym:14}  signals={len(v["signals"]):4d}  orders={len(v["orders"]):4d}  closes={len(v["closes"]):4d}  cycles={len(v["cycles"]):5d}')


Loaded live events — signals=0  orders=0  closes=0
  ADAUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  AVAXUSDT        signals=   0  orders=   0  closes=   0  cycles=    0
  BCHUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  BNBUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  BTCUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  DOGEUSDT        signals=   0  orders=   0  closes=   0  cycles=    0
  DOTUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  ETHUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  LTCUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  SHIB1000USDT    signals=   0  orders=   0  closes=   0  cycles=    0
  SOLUSDT         signals=   0  orders=   0  closes=   0  cycles=    2
  XAUUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  XLMUSDT         signals=   0  orders=   0  closes=   0  cycles=    0
  XRPUSDT         signals=

## Section 4 — Replay backtest on the same window

We invoke `run_strategy()` from `_strategy_lib.py` with each symbol's winner config and
filter the resulting trades to the live window. **This is the same backtest engine the
sweep used** — identical fees, identical SL/TP, identical signal pipeline.

In [4]:
def replay_one(symbol: str, window_from: pd.Timestamp, window_to: pd.Timestamp) -> dict:
    cfg_path = WINNERS_DIR / symbol / 'config.json'
    if not cfg_path.exists():
        return dict(skipped=f'no winner config for {symbol}')
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    sname = cfg['strategy']
    fn = STRATEGY_FN.get(sname)
    if fn is None:
        return dict(skipped=f'unknown strategy {sname}')
    params = dict(cfg.get('params', {}))
    if 'session' in params and isinstance(params['session'], list):
        params['session'] = tuple(params['session'])
    if 'confirms' in params and isinstance(params['confirms'], list):
        params['confirms'] = tuple(params['confirms'])
    tf  = cfg['base_tf']
    htf = cfg.get('htf', 'H1' if tf == 'M5' else 'H4')
    use_daily = bool(cfg.get('use_daily', USE_DAILY.get(sname, False)))
    rr = float(cfg.get('rr', 0.0) or 0.0)
    # We pad backwards 60 days for indicator warmup so the early signals
    # use the same RSI/ATR state the live bot saw.
    pad_from = (window_from - pd.Timedelta(days=60)).strftime('%Y-%m-%d')
    date_to  = window_to.strftime('%Y-%m-%d')
    kw = dict(use_daily=use_daily, max_hold_bars=96 if tf == 'M5' else 48,
              date_from=pad_from, date_to=date_to)
    if rr > 0:
        kw['rr'] = rr
    trades, df_sig = run_strategy(symbol, tf, htf, fn, params, **kw)
    if df_sig.empty:
        return dict(skipped=f'no data for {symbol}')
    # Trade rows that opened inside the live window
    rows = []
    for t in trades:
        if t.entry_time is None: continue
        et = pd.Timestamp(t.entry_time)
        if et < window_from or et >= window_to: continue
        rows.append({
            'entry_time': et, 'side': 'long' if t.side == 1 else 'short',
            'entry': t.entry, 'sl': t.sl, 'tp': t.tp,
            'exit_time': pd.Timestamp(t.exit_time) if t.exit_time is not None else pd.NaT,
            'exit': t.exit, 'reason': t.reason,
            'R': round(t.r_multiple, 3), 'net_R': round(t.net_r, 3),
        })
    trades_df = pd.DataFrame(rows)
    # Every bar where df_sig.signal != 0 inside the window (signals, not just trades)
    df_sig = df_sig.copy()
    df_sig['time'] = pd.to_datetime(df_sig['time'])
    sig_rows = df_sig[(df_sig.get('signal', 0) != 0) &
                      (df_sig['time'] >= window_from) &
                      (df_sig['time'] <  window_to)]
    signals_df = sig_rows[['time', 'signal', 'close', 'sl_price']].copy()
    if 'tp_price' in sig_rows.columns:
        signals_df['tp_price'] = sig_rows['tp_price']
    return dict(trades=trades_df, signals=signals_df, cfg=cfg)

replay_by_sym: dict[str, dict] = {}
for sym in active:
    try:
        replay_by_sym[sym] = replay_one(sym, T_FROM, T_TO)
    except Exception as exc:
        replay_by_sym[sym] = {'skipped': f'exception: {exc!r}'}
for sym, r in replay_by_sym.items():
    if 'skipped' in r:
        print(f'  {sym:14}  SKIP — {r["skipped"]}')
    else:
        print(f'  {sym:14}  replay  signals={len(r["signals"]):4d}  trades={len(r["trades"]):4d}')


  ADAUSDT         replay  signals=  15  trades=   0
  AVAXUSDT        replay  signals=  14  trades=   1
  BCHUSDT         replay  signals=   0  trades=   0
  BNBUSDT         replay  signals=   0  trades=   0
  BTCUSDT         replay  signals=   0  trades=   0
  DOGEUSDT        replay  signals=  13  trades=   0
  DOTUSDT         replay  signals=   7  trades=   1
  ETHUSDT         replay  signals=   0  trades=   0
  LTCUSDT         replay  signals=   0  trades=   0
  SHIB1000USDT    replay  signals=  15  trades=   1
  SOLUSDT         replay  signals=  16  trades=   1
  XAUUSDT         replay  signals=   0  trades=   0
  XLMUSDT         replay  signals=   0  trades=   0
  XRPUSDT         replay  signals=   0  trades=   0


## Section 5 — Match live ⇄ replay signals

For each symbol we join the live `signal` events with replay-produced signals by
(symbol, bar_time, direction). Categorise each row as:

- `matched`      — live + replay agree on the bar and direction.
- `live_only`    — bot fired a signal the backtest didn't reproduce. → drift bug.
- `replay_only`  — backtest expected a signal the bot missed.        → live miss / outage.

All matched / mismatched rows get a `delta_entry` (live entry − replay close)
and a `delta_sl/tp` column so price-level drift is obvious.

In [5]:
def _to_ts(x):
    try:
        t = pd.Timestamp(x)
    except Exception:
        return pd.NaT
    if t is pd.NaT: return pd.NaT
    if t.tzinfo is not None:
        t = t.tz_convert('UTC').tz_localize(None)
    return t

def signals_diff(symbol: str) -> pd.DataFrame:
    live = live_by_sym.get(symbol, {}).get('signals', pd.DataFrame())
    rep  = replay_by_sym.get(symbol, {}).get('signals', pd.DataFrame())
    if live.empty and rep.empty:
        return pd.DataFrame()
    if not live.empty:
        live = live.copy()
        live['bar_dt']    = live['bar_time'].apply(_to_ts)
        live['dir_str']   = live['direction']
        live['side_int']  = np.where(live['direction'] == 'long', 1, -1)
    if not rep.empty:
        rep = rep.copy()
        rep['bar_dt']    = rep['time'].apply(_to_ts)
        rep['side_int']  = rep['signal'].astype(int)
        rep['dir_str']   = np.where(rep['side_int'] == 1, 'long', 'short')

    tol = pd.Timedelta(minutes=BAR_TOLERANCE_MIN)
    rows = []
    used_rep = set()
    if not live.empty:
        for _, lr in live.iterrows():
            match = None
            if not rep.empty:
                cand = rep[(rep['side_int'] == lr['side_int']) &
                           (rep['bar_dt'] >= lr['bar_dt'] - tol) &
                           (rep['bar_dt'] <= lr['bar_dt'] + tol)]
                cand = cand[~cand.index.isin(used_rep)]
                if not cand.empty:
                    match = cand.iloc[0]
                    used_rep.add(match.name)
            rows.append({
                'symbol':       symbol,
                'kind':         'matched' if match is not None else 'live_only',
                'bar_time':     lr['bar_dt'],
                'direction':    lr['dir_str'],
                'live_entry':   float(lr.get('entry', np.nan)),
                'live_sl':      float(lr.get('sl',    np.nan)),
                'live_tp':      float(lr.get('tp',    np.nan)),
                'rep_close':    float(match['close'])    if match is not None else np.nan,
                'rep_sl':       float(match['sl_price']) if match is not None else np.nan,
                'rep_tp':       float(match['tp_price']) if (match is not None and 'tp_price' in match.index) else np.nan,
            })
    if not rep.empty:
        unmatched = rep[~rep.index.isin(used_rep)]
        for _, rr in unmatched.iterrows():
            rows.append({
                'symbol':       symbol,
                'kind':         'replay_only',
                'bar_time':     rr['bar_dt'],
                'direction':    rr['dir_str'],
                'live_entry':   np.nan,
                'live_sl':      np.nan,
                'live_tp':      np.nan,
                'rep_close':    float(rr['close']),
                'rep_sl':       float(rr['sl_price']),
                'rep_tp':       float(rr['tp_price']) if 'tp_price' in rr.index else np.nan,
            })
    out = pd.DataFrame(rows).sort_values('bar_time') if rows else pd.DataFrame()
    if not out.empty:
        out['delta_entry'] = out['live_entry'] - out['rep_close']
        out['delta_sl']    = out['live_sl']    - out['rep_sl']
        out['delta_tp']    = out['live_tp']    - out['rep_tp']
    return out

diff_frames = [signals_diff(sym) for sym in active]
diff_all = pd.concat([d for d in diff_frames if not d.empty], ignore_index=True) if diff_frames else pd.DataFrame()
print(f'Total diff rows: {len(diff_all)}')
if not diff_all.empty:
    print(diff_all['kind'].value_counts())
    diff_all.head(40)


Total diff rows: 80
kind
replay_only    80
Name: count, dtype: int64


## Section 6 — Per-symbol parity scoreboard

How well did live track replay over the window? For each symbol we report
`matched / live_only / replay_only` counts + the win rate / net R of the live
closes versus the replay trades.

In [6]:
rows = []
for sym in active:
    diff = next((d for d in diff_frames if not d.empty and d['symbol'].iloc[0] == sym), pd.DataFrame())
    n_matched     = int((diff['kind'] == 'matched').sum())     if not diff.empty else 0
    n_live_only   = int((diff['kind'] == 'live_only').sum())   if not diff.empty else 0
    n_replay_only = int((diff['kind'] == 'replay_only').sum()) if not diff.empty else 0
    live_closes = live_by_sym.get(sym, {}).get('closes', pd.DataFrame())
    if not live_closes.empty:
        live_wins = int((live_closes['pnl_usdt'].astype(float) > 0).sum()) if 'pnl_usdt' in live_closes.columns else 0
        live_n    = len(live_closes)
        live_net  = float(live_closes['pnl_usdt'].astype(float).sum())     if 'pnl_usdt' in live_closes.columns else 0.0
    else:
        live_wins, live_n, live_net = 0, 0, 0.0
    rep_trades = replay_by_sym.get(sym, {}).get('trades', pd.DataFrame())
    if not rep_trades.empty:
        rep_wins  = int((rep_trades['R'] > 0).sum())
        rep_n     = len(rep_trades)
        rep_net_R = float(rep_trades['net_R'].sum())
    else:
        rep_wins, rep_n, rep_net_R = 0, 0, 0.0
    rows.append({
        'symbol': sym,
        'live_closed':   live_n, 'live_wins':   live_wins,
        'live_WR_%':     round(100 * live_wins / live_n, 1) if live_n else 0.0,
        'live_net_USDT': round(live_net, 2),
        'replay_trades': rep_n, 'replay_wins': rep_wins,
        'replay_WR_%':   round(100 * rep_wins / rep_n, 1) if rep_n else 0.0,
        'replay_net_R':  round(rep_net_R, 3),
        'matched':      n_matched,
        'live_only':    n_live_only,
        'replay_only':  n_replay_only,
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


      symbol  live_closed  live_wins  live_WR_%  live_net_USDT  replay_trades  replay_wins  replay_WR_%  replay_net_R  matched  live_only  replay_only
     ADAUSDT            0          0   0.000000       0.000000              0            0     0.000000      0.000000        0          0           15
    AVAXUSDT            0          0   0.000000       0.000000              1            0     0.000000     -1.095000        0          0           14
     BCHUSDT            0          0   0.000000       0.000000              0            0     0.000000      0.000000        0          0            0
     BNBUSDT            0          0   0.000000       0.000000              0            0     0.000000      0.000000        0          0            0
     BTCUSDT            0          0   0.000000       0.000000              0            0     0.000000      0.000000        0          0            0
    DOGEUSDT            0          0   0.000000       0.000000              0            0    

## Section 7 — Persist artefacts

Everything saves to `notebooks/data/` tagged with the window so multiple replays don't
stomp each other.

In [7]:
def concat_live(field: str) -> pd.DataFrame:
    frames = []
    for sym, v in live_by_sym.items():
        df = v.get(field, pd.DataFrame())
        if df.empty: continue
        df = df.copy(); df['symbol'] = sym
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

live_signals = concat_live('signals')
live_orders  = concat_live('orders')
live_closes  = concat_live('closes')

replay_signals_all = pd.concat(
    [r['signals'].assign(symbol=s) for s, r in replay_by_sym.items()
     if not r.get('skipped') and not r['signals'].empty],
    ignore_index=True
) if any(not r.get('skipped') for r in replay_by_sym.values()) else pd.DataFrame()

out_files = {
    f'live_signals_{TAG}.csv':         live_signals,
    f'live_orders_{TAG}.csv':          live_orders,
    f'live_closes_{TAG}.csv':          live_closes,
    f'replay_signals_{TAG}.csv':       replay_signals_all,
    f'replay_vs_live_diff_{TAG}.csv':  diff_all,
    f'replay_summary_{TAG}.csv':       summary,
}
for fname, df in out_files.items():
    if df.empty:
        print(f'  {fname:42s}  (empty — not written)')
        continue
    p = OUT_DIR / fname
    df.to_csv(p, index=False)
    print(f'  {fname:42s}  rows={len(df):5d}  -> {p}')


  live_signals_20260530_20261231.csv          (empty — not written)
  live_orders_20260530_20261231.csv           (empty — not written)
  live_closes_20260530_20261231.csv           (empty — not written)
  replay_signals_20260530_20261231.csv        rows=   80  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_signals_20260530_20261231.csv
  replay_vs_live_diff_20260530_20261231.csv   rows=   80  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_vs_live_diff_20260530_20261231.csv
  replay_summary_20260530_20261231.csv        rows=   14  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_summary_20260530_20261231.csv


## Section 8 — Drilldown on a single mismatch

Pick a row from `replay_vs_live_diff_*.csv` and look at the *full cycle log* the bot
wrote around that bar. The `cycle` event contains every gate value the strategy fn
evaluated (RSI/ADX/h1_trend/MACD), so any disagreement vs the replay's `df_sig` row
tells you *which gate* drifted (data freshness, indicator value, etc.).

In [8]:
def explain(sym: str, bar_time_str: str, window_min: int = 30) -> pd.DataFrame:
    cycles = live_by_sym.get(sym, {}).get('cycles', pd.DataFrame())
    if cycles.empty:
        return pd.DataFrame()
    bar = pd.Timestamp(bar_time_str)
    cycles = cycles.copy()
    cycles['ts'] = pd.to_datetime(cycles['ts'])
    near = cycles[(cycles['ts'] >= bar - pd.Timedelta(minutes=window_min)) &
                   (cycles['ts'] <= bar + pd.Timedelta(minutes=window_min))]
    return near

# Example: change SYM and TIME below to inspect the gates the bot saw.
if not diff_all.empty and (diff_all['kind'] != 'matched').any():
    first = diff_all[diff_all['kind'] != 'matched'].iloc[0]
    print(f'Drilldown candidate: {first["symbol"]} bar={first["bar_time"]}  kind={first["kind"]}')
    near = explain(first['symbol'], str(first['bar_time']))
    if near.empty:
        print('No cycle events recorded near this time. Was the bot running?')
    else:
        # Keep the columns most useful for forensic comparison.
        keep = [c for c in ('ts','last_bar','m5_bars','htf_bars','age_min',
                              'signal','diag') if c in near.columns]
        print(near[keep].to_string(index=False))
else:
    print('Either no diffs found, or all rows are "matched". Pick a different window.')


Drilldown candidate: ADAUSDT bar=2026-05-30 00:35:00  kind=replay_only
No cycle events recorded near this time. Was the bot running?
